In [4]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))  ##LETS THE NOTEBOOK FIND THE SRC/FOLDER
from src.data_utils import load_phrasebank, repair_mojibake, MOJIBAKE_MAP

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "FinancialPhraseBank-v1.0" / "Sentences_75Agree.txt"
df, bad_lines = load_phrasebank(DATA_FILE)
pd.set_option("display.max_colwidth", None)
print("Rows",len(df),"| Map entries:",len(MOJIBAKE_MAP))

Rows 3453 | Map entries: 64


In [8]:
import importlib
import src.data_utils as du
importlib.reload(du)                      # force Python to re-read the file from disk

print("File in use:", du.__file__)
print("Map entries:", len(du.MOJIBAKE_MAP))
print("Has rule 2:", "-®" in du.MOJIBAKE_MAP)

repair_mojibake = du.repair_mojibake      # make sure Cell 2 uses the freshly loaded version

File in use: h:\Desktop\financial-news-intelligence\src\data_utils.py
Map entries: 97
Has rule 2: True


In [10]:
# repair and check
non_ascii = r"[^\x00-\x7F]"
before = df ["sentence"].str.contains(non_ascii).sum()

df["sentence_clean"]=df["sentence"].apply(repair_mojibake)
changed = df["sentence"] != df["sentence_clean"]
after = df["sentence_clean"].str.contains(r"[+-][^\x00-\x7F]").sum()

print("Sentence with non-ASCII before:",before)
print("Sentence repaired:", changed.sum())
print("Sentence still containing a garbled-looking pair:", after)
print()
for old, new in df.loc[changed, ["sentence","sentence_clean"]].head(6).values:
    print("Before:",old)
    print("After:",new,"\n")

print("Still unrepaired:")
for s in df.loc[df["sentence_clean"].str.contains(r"[+-][^\x00-\x7F]"),"sentence_clean"]:
    print("-",s)

Sentence with non-ASCII before: 74
Sentence repaired: 74
Sentence still containing a garbled-looking pair: 0

Before: Clothing retail chain Sepp+ñl+ñ 's sales increased by 8 % to EUR 155.2 mn , and operating profit rose to EUR 31.1 mn from EUR 17.1 mn in 2004 .
After: Clothing retail chain Seppälä 's sales increased by 8 % to EUR 155.2 mn , and operating profit rose to EUR 31.1 mn from EUR 17.1 mn in 2004 . 

Before: Finnish Bank of +àland 's consolidated net operating profit increased from EUR 4.8 mn in the first quarter of 2005 to EUR 6.4 mn in the first quarter of 2006 .
After: Finnish Bank of Åland 's consolidated net operating profit increased from EUR 4.8 mn in the first quarter of 2005 to EUR 6.4 mn in the first quarter of 2006 . 

Before: Net sales of Finnish food industry company L+ñnnen Tehtaat 's continuing operations increased by 13 % in 2008 to EUR 349.1 mn from EUR 309.6 mn in 2007 .
After: Net sales of Finnish food industry company Lännen Tehtaat 's continuing operations

In [22]:
## apply, inspect, drop duplicates
importlib.reload(du)
df["sentence_clean"] = df["sentence_clean"].apply(du.normalize_numbers)

## every sentence the number step touched, so we can check it by eye
touched = df["sentence"].apply(du.repair_mojibake) != df["sentence_clean"]
print("Sentences changed by number normalization:", touched.sum())
for s in df.loc[touched, "sentence_clean"].sample(8, random_state=42):
    print(" -", s)

## rule-5 risk: "m" after a number might be metres, not million
print("\nPossible false 'million':")
for s in df.loc[df["sentence"].str.contains(r"\d m\b"), "sentence"]:
    print(" -", s)

## exact duplicates: drop them now, on the CLEAN text
before = len(df)
df = df.drop_duplicates(subset="sentence_clean").reset_index(drop=True)
print(f"\nDropped {before - len(df)} duplicates -> {len(df)} rows")

Sentences changed by number normalization: 710
 - The stock is trading above both its MAPs and the 50-day MAP of SEK 72.983 is higher than the 200-day MAP of SEK 70.283 , a bullish indicator .
 - SKF 6 April 2010 - Alandsbanken has given a `` buy '' recommendation on Swedish industrial company SKF AB ( STO : SKF B ) with a share price target of SEK 150 .
 - Operating profit excluding non-recurring items decreased to EUR 6.2 million from EUR 16.8 million in 2007 , representing 2.3 % of net sales .
 - The company reports a loss for the period of EUR 0.4 million compared to a loss of EUR 1.9 million in the corresponding period in 2005 .
 - Finnish handling systems company Cargotec Oyj ( HEL : CGCBV ) said today that it won a EUR 13 million ( USD 16.6 million ) contract to deliver MacGregor hatch covers for ships ordered by Norwegian shipowner Grieg Shipping .
 - Separately , YIT Corporation and Austrian firm E AG last week signed an agreement on the sale of E 's building system service bu

In [23]:
raw, _ = du.load_phrasebank(DATA_FILE)
raw["sentence_clean"] = raw["sentence"].apply(du.repair_mojibake).apply(du.normalize_numbers)
dup = raw[raw.duplicated("sentence_clean", keep=False)]
print("Groups with conflicting labels:", (dup.groupby("sentence_clean")["label"].nunique() > 1).sum())
for key, g in dup.groupby("sentence_clean"):
    if g["sentence"].nunique() > 1:          # groups that only became duplicates AFTER cleaning
        print()
        print(g[["sentence", "label"]].to_string(index=False))

Groups with conflicting labels: 0

                                                                                    sentence   label
For 2009 , net profit was EUR 3 million and the company paid a dividend of EUR 1.30 apiece . neutral
        For 2009 , net profit was EUR3m and the company paid a dividend of EUR1 .30 apiece . neutral

                                                                                                                                          sentence    label
However , the company saw its net profit for the third quarter down to EUR 1.4 million from EUR 1.5 million for the corresponding period of 2009 . negative
            However , the company saw its net profit for the third quarter down to EUR1 .4 m from EUR1 .5 m for the corresponding period of 2009 . negative

                                                                                                                 sentence    label
In the second quarter of 2010 , the group 's net profit rose 

In [24]:
from sklearn.model_selection import StratifiedGroupKFold
importlib.reload(du)

# 1. Give every sentence its template key
df["group"] = df["sentence_clean"].apply(du.make_group_key)

# 2. Cut off the TEST pile (1 of 7 slices, about 14%)
splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
rest_idx, test_idx = next(splitter.split(df, df["label"], groups=df["group"]))
test_df = df.iloc[test_idx]
rest_df = df.iloc[rest_idx]

# 3. Cut the VALIDATION pile from what's left (1 of 6 slices, about 14%)
splitter2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=42)
train_idx, val_idx = next(splitter2.split(rest_df, rest_df["label"], groups=rest_df["group"]))
train_df = rest_df.iloc[train_idx]
val_df = rest_df.iloc[val_idx]

# 4. Check rule 1: same label mix in every pile
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    mix = part["label"].value_counts(normalize=True).mul(100).round(1).to_dict()
    print(f"{name:>5}: {len(part):>4} rows ({len(part) / len(df) * 100:.1f}%) | {mix}")

# 5. Check rule 2: no template appears in two piles (all three numbers must be 0)
tr, va, te = set(train_df["group"]), set(val_df["group"]), set(test_df["group"])
print("\nShared templates  train-val:", len(tr & va), "| train-test:", len(tr & te), "| val-test:", len(va & te))

# 6. Save the piles, so ALL 5 models use the exact same split
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part[["sentence_clean", "label"]].to_csv(OUT_DIR / f"{name}.csv", index=False, encoding="utf-8")
print("\nSaved to", OUT_DIR)

train: 2456 rows (71.4%) | {'neutral': 62.1, 'positive': 25.7, 'negative': 12.2}
  val:  492 rows (14.3%) | {'neutral': 62.2, 'positive': 25.6, 'negative': 12.2}
 test:  492 rows (14.3%) | {'neutral': 62.2, 'positive': 25.8, 'negative': 12.0}

Shared templates  train-val: 0 | train-test: 0 | val-test: 0

Saved to h:\Desktop\financial-news-intelligence\data\processed
